In [2]:
import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations
import time
import itertools

In [10]:
n_user = 2
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
# h = [[0.1001, 0.1], [0.1001, 1], [1.001, 0.1], [1.001, 1]]
#############################################################
rho = np.arange(0, r_max+1, 1)
P_bar_U = n_user*[5]
W = list(product(rho, repeat=n_user))
print(W)
print('W', W)
mu = cp.Variable((len(h), len(W) ), nonneg=True)
Pow = cp.Variable((n_user, len(h), len(W)), nonneg=True )
p = []
for x in range(n_user):
    temp  = 0
    for k in range(len(h)):
        for j in range(len(W)):
            if W[j][x] != 0:
                temp+= mu[k][j] * hp[k]
    p.append(temp)
P_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(W)):
            temp+= Pow[x][k][j]  *  hp[k]
    P_u.append(temp)
obj_fun = 0
for n in range(n_user):
    obj_fun += ((cp.inv_pos(p[n]) - 1) )
objective = cp.Minimize(obj_fun)
constraints = []
for n in range(n_user):
    constraints.append(P_u[n] <= P_bar_U[n] )
a_val = []
for q in range(len(h)):
    a_val.append( cp.sum([ mu[q][j] for j in range(len(W))]))
for a in a_val:
    constraints.append(a == 1)
for i in range(len(h)):           
    for j in range(len(W)):
        constraints.append(0<= mu[i][j])
        constraints.append(mu[i][j] <= 1)

users = list(range(n_user))
for k in range(len(h)):
    for j in range(len(W)):
        for r in range(1, n_user + 1): 
            for S in itertools.combinations(users, r):
                sum_rate = sum(W[j][x] for x in S)
                sum_power = sum(Pow[x][k][j] * h[k][x] for x in S)
                temp1 = mu[k][j] * (2**sum_rate - 1)
                temp2 = sum_power
                constraints.append(temp1 <= temp2)

prob = cp.Problem(objective, constraints)
prob.solve()
print('age', np.round(prob.value, 4))
print('mu opt\n', np.round(mu.value, 3))
print('pi1',  np.round(Pow[0].value, 3))
print('pi2', np.round(Pow[1].value, 3))
for n in range(n_user):
    print('Pow',n+1,':',  np.round(P_u[n].value, 3), 'p', n+1, ':', np.round(p[n].value, 5))

[(np.int64(0), np.int64(0)), (np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(1), np.int64(1))]
W [(np.int64(0), np.int64(0)), (np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(1), np.int64(1))]
age 0.2378
mu opt
 [[0.    0.425 0.425 0.15 ]
 [0.    0.    0.    1.   ]
 [0.    0.    0.    1.   ]
 [0.    0.    0.    1.   ]]
pi1 [[ 0.    0.    4.25  2.25]
 [ 0.    0.    0.   10.  ]
 [ 0.    0.    0.    2.  ]
 [ 0.    0.    0.    1.5 ]]
pi2 [[ 0.    4.25  0.    2.25]
 [ 0.    0.    0.    2.  ]
 [ 0.    0.    0.   10.  ]
 [ 0.    0.    0.    1.5 ]]
Pow 1 : 5.0 p 1 : 0.89375
Pow 2 : 5.0 p 2 : 0.89375


/home/sysad/.local/lib/python3.10/site-packages/cvxpy/reductions/solvers/solving_chain_utils.py:41: UserWarning: The problem has an expression with dimension greater than 2. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(


# SOlve for F

In [17]:
mu_opt = mu.value
F = cp.Variable((n_user, len(h), len(W)), nonneg=True )
p = []
for x in range(n_user):
    temp  = 0
    for k in range(len(h)):
        for j in range(len(W)):
            if W[j][x] != 0:
                temp+= mu_opt[k][j] * hp[k]
    p.append(temp)
P_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(W)):
            temp+= F[x][k][j]  *  hp[k] * mu_opt[k][j]
    P_u.append(temp)
obj_fun = 0
for n in range(n_user):
    obj_fun += ((cp.inv_pos(p[n]) - 1) )
objective = cp.Minimize(obj_fun)
constraints = []
for n in range(n_user):
    constraints.append(P_u[n] <= P_bar_U[n] )

users = list(range(n_user))
for k in range(len(h)):
    for j in range(len(W)):
        for r in range(1, n_user + 1): 
            for S in itertools.combinations(users, r):
                sum_rate = sum(W[j][x] for x in S)
                sum_power = sum(F[x][k][j] * h[k][x] for x in S)
                temp1 = (2**sum_rate - 1)
                temp2 = sum_power
                constraints.append(temp1 <= temp2)

prob = cp.Problem(objective, constraints)
prob.solve()
print('F1',  np.round(F[0].value, 3))
print('F2', np.round(F[1].value, 3))
print(np.round((P_u[0].value, P_u[1].value), 3))

F1 [[ 2.782  0.    10.    15.153]
 [ 4.134  0.    11.083 10.   ]
 [ 4.014  4.015  1.     2.   ]
 [ 3.475  3.455  3.961  1.5  ]]
F2 [[ 2.782 10.     0.    15.153]
 [ 4.014  1.     4.015  2.   ]
 [ 4.134 11.083  0.    10.   ]
 [ 3.475  3.961  3.455  1.5  ]]
[4. 4.]


# Time Sharing

In [20]:
n_user = 3
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
rho = np.arange(0, r_max+1, 1)
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
W = list(product(rho, repeat=n_user))
# W = [(0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)]
# W = [(0, 0), (1, 0), (0, 1)]
rho_DO = []
# poss_dec_ord = [(1, 2), (2, 1)]
poss_dec_ord = [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
for w in W:
    for DO in poss_dec_ord:
        rho_DO.append([w, DO])
print(rho_DO)
def cal_pow_TS(R, DO, h_val):
    pow_vals = np.zeros(len(DO))
    for i in range(len(DO)):
        idx = DO[i]-1
        sum_i_M = np.sum([R[DO[k]-1]  for k in range(i, len(DO))])
        sum_ip1_M = np.sum([R[DO[k]-1] for k in range(min(i+1, len(DO)), len(DO))])
        pow = (2**(sum_i_M) - 2**(sum_ip1_M))/h_val[idx]
        pow_vals[idx] = pow
    return pow_vals
###########################################################3
P_bar_U = n_user*[5]
mu = cp.Variable((len(h), len(rho_DO) ), nonneg=True)
p = []
for x in range(n_user):
    temp  = 0
    for k in range(len(h)):
        for j in range(len(rho_DO)):
            if rho_DO[j][0][x] != 0:
                temp+= mu[k][j] * hp[k]
    p.append(temp)
P_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(rho_DO)):
            temp+= cal_pow_TS(rho_DO[j][0], rho_DO[j][1] , h[k])[x] * mu[k][j] *  hp[k]
    P_u.append(temp)
obj_fun = 0
for n in range(n_user):
    obj_fun = obj_fun + ( (cp.inv_pos(p[n])  -  1 )  )
objective = cp.Minimize(obj_fun)
constraints = []
for n in range(n_user):
    constraints.append(P_u[n] <= P_bar_U[n] )
a_val = []
for q in range(len(h)):
    a_val.append( cp.sum([ mu[q][j] for j in range(len(rho_DO))]))
for a in a_val:
    constraints.append(a == 1)
for i in range(len(h)):           
    for j in range(len(rho_DO)):
        constraints.append(0<= mu[i][j])
        constraints.append(mu[i][j] <= 1)
prob = cp.Problem(objective, constraints)
prob.solve()
print('age', np.round(prob.value, 4))
# print('mu opt\n', np.round(mu.value, 3))
for n in range(n_user):
    print('Pow',n,':',  np.round(P_u[n].value, 3))

[[(np.int64(0), np.int64(0), np.int64(0)), (1, 2, 3)], [(np.int64(0), np.int64(0), np.int64(0)), (1, 3, 2)], [(np.int64(0), np.int64(0), np.int64(0)), (2, 1, 3)], [(np.int64(0), np.int64(0), np.int64(0)), (2, 3, 1)], [(np.int64(0), np.int64(0), np.int64(0)), (3, 1, 2)], [(np.int64(0), np.int64(0), np.int64(0)), (3, 2, 1)], [(np.int64(0), np.int64(0), np.int64(1)), (1, 2, 3)], [(np.int64(0), np.int64(0), np.int64(1)), (1, 3, 2)], [(np.int64(0), np.int64(0), np.int64(1)), (2, 1, 3)], [(np.int64(0), np.int64(0), np.int64(1)), (2, 3, 1)], [(np.int64(0), np.int64(0), np.int64(1)), (3, 1, 2)], [(np.int64(0), np.int64(0), np.int64(1)), (3, 2, 1)], [(np.int64(0), np.int64(1), np.int64(0)), (1, 2, 3)], [(np.int64(0), np.int64(1), np.int64(0)), (1, 3, 2)], [(np.int64(0), np.int64(1), np.int64(0)), (2, 1, 3)], [(np.int64(0), np.int64(1), np.int64(0)), (2, 3, 1)], [(np.int64(0), np.int64(1), np.int64(0)), (3, 1, 2)], [(np.int64(0), np.int64(1), np.int64(0)), (3, 2, 1)], [(np.int64(0), np.int64(1),

# Fix Pow values - SIC

In [ ]:
# def cal_pow(R, beta, h_val):
#     M = len(R)
#     order = np.argsort(np.array(beta) / np.array(h_val))
#     P = np.zeros(M)
#     for i in range(M):
#         ui = order[i]
#         sum_i_M = sum(R[order[k]] for k in range(i, M))
#         sum_ip1_M = sum(R[order[k]] for k in range(i+1, M))
#         P[ui] = (2**sum_i_M - 2**sum_ip1_M) / h_val[ui]
#     # print(h_val, order, P, R)
#     return P


# n_user = 2
# B = [1, 3]
# h_bad =  0.1
# h_good = 1
# pr_h_bad = 0.5
# pr_h_good = (1- pr_h_bad)
# r_max = 1
# h_vec_u1 = np.array([h_bad,  h_good]).tolist()
# permutations_list = []
# permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
# combined_h_vec = []
# for permutation in permutations_list:
#     arr = []
#     for zz in permutation:
#         arr.append(zz)
#     combined_h_vec.append(arr)
# p_h_u1 = np.array([pr_h_bad, pr_h_good])
# permutations_list = []
# permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
# pr_h = []
# for permutation in permutations_list:
#     arr = []
#     for k in permutation:
#         arr.append(k)
#     pr_h.append(np.prod(arr))
# hp = pr_h
# h = combined_h_vec
# # h = [[0.1001, 0.1], [0.1001, 1], [1.001, 0.1], [1.001, 1]]
# print(h)
# #####################################################
# rho = np.arange(0, r_max+1, 1)
# P_bar_U = [5, 5]
# W = list(product(rho, repeat=n_user))
# print(W)
# mu = cp.Variable((len(h), len(W) ), nonneg=True)
# p = []
# for x in range(n_user):
#     temp  = 0
#     for k in range(len(h)):
#         for j in range(len(W)):
#             if W[j][x] != 0:
#                 temp+= mu[k][j] * hp[k]
#     p.append(temp)
# P_u = []
# for x in range(n_user):
#     temp = 0
#     for k in range(len(h)):
#         for j in range( len(W)):
#             temp+= cal_pow(W[j], B, h[k])[x] * mu[k][j] *  hp[k]
#         # print('-----------------')
#     P_u.append(temp)
#     # print('-------------')
# obj_fun = 0
# for n in range(n_user):
#     obj_fun += ( (cp.inv_pos(p[n]) - 1) )
# objective = cp.Minimize(obj_fun)
# constraints = []
# for n in range(n_user):
#     constraints.append(P_u[n] <= P_bar_U[n] )
# a_val = []
# for q in range(len(h)):
#     a_val.append( cp.sum([ mu[q][j] for j in range(len(W))]))
# for a in a_val:
#     constraints.append(a == 1)
# for i in range(len(h)):           
#     for j in range(len(W)):
#         constraints.append(0<= mu[i][j])
#         constraints.append(mu[i][j] <= 1)
# prob = cp.Problem(objective, constraints)
# prob.solve()
# print('age', np.round(prob.value, 4))
# print('mu', np.round(mu.value, 3))
# for n in range(n_user):
#     print('Pow',n,':',  np.round(P_u[n].value, 3))
#     print('p', n, ':', np.round(p[n].value, 3))